In [0]:
%run  ../init/kafka_config

In [0]:
from confluent_kafka import Producer
import json
import uuid
import random

In [0]:
producer = Producer(producer_conf)

In [0]:
def delivery_report(err, msg):
    if err:
        print(f"✗ Failed: {err}")
    else:
        print(f"""✓ Order sent → topic: {msg.topic()} | partition: {msg.partition()} | offset: {msg.offset()}""")

In [0]:
# ── Sample user and item pools ─────────────
# These match what we inserted in insert_sample_data
user_pool = [
    ("U001", "normal"),
    ("U002", "normal"),
    ("U003", "normal"),
    ("U004", "premium"),
    ("U005", "normal"),
]

item_pool = ["I001", "I002", "I003", "I004", "I005"]

In [0]:
def custom_partitioner(key_bytes, all_partitions, available_partitions):
    key = key_bytes.decode("utf-8")
    if key == "premium":
        return 2
    else:
        return random.choice([0, 1])

In [0]:
# ── Generate and Send Orders ───────────────
num_orders = 10

producer = Producer(producer_conf)  # normal producer, no custom partitioner

print(f"Sending {num_orders} orders to Kafka...\n")

for _ in range(num_orders):
    user_id, category = random.choice(user_pool)

    order = {
        "order_id":      str(uuid.uuid4()),
        "user_id":       user_id,
        "item_id":       random.choice(item_pool),
        "item_quantity": random.randint(1, 5),
        "category":      category
    }

    # decide partition directly
    if category == "premium":
        partition = 2
    else:
        partition = random.choice([0, 1])

    producer.produce(
        topic="orders",
        value=json.dumps(order).encode("utf-8"),
        key=category.encode("utf-8"),
        partition=partition,              # ← directly tell Kafka which partition
        callback=delivery_report
    )
    producer.poll(0)

producer.flush()
print("\n✓ All orders sent successfully")